# Kiểm tra và phân tích dữ liệu văn bản hành chính

Notebook này được sử dụng để kiểm tra và phân tích hai nguồn dữ liệu chính của dự án:

1. **Bộ mẫu văn bản hành chính (forms)** được lưu dưới dạng Markdown.
2. **Bộ dữ liệu văn bản pháp luật / hành chính** được lưu dưới dạng Parquet.

Mục tiêu của notebook là giúp nhóm dự án:

- Hiểu cấu trúc dữ liệu
- Kiểm tra tính đầy đủ và tính nhất quán của dữ liệu
- Xác minh schema trước khi đưa dữ liệu vào các bước xử lý tiếp theo

Kết quả của quá trình phân tích này sẽ giúp đảm bảo dữ liệu sẵn sàng cho các bước như:

- Xây dựng hệ thống RAG (Retrieval Augmented Generation)
- Trích xuất thông tin từ văn bản
- Sinh văn bản hành chính từ template

## Tổng quan về các bộ dữ liệu

Dự án sử dụng hai loại dữ liệu khác nhau:

### 1. Bộ mẫu văn bản hành chính (Markdown Forms)

Đây là các **mẫu văn bản hành chính chuẩn**, được chuyển sang định dạng Markdown.

Các mẫu văn bản bao gồm:

- Nghị quyết
- Quyết định
- Công văn
- Công điện
- Giấy mời
- Biên bản
- Giấy giới thiệu
- Giấy nghỉ phép

Mỗi file Markdown gồm:

- Phần **metadata (YAML frontmatter)** mô tả loại văn bản
- Nội dung **template văn bản**
- Các **placeholder** dùng để điền thông tin thực tế

Ví dụ placeholder:


## Quy trình thực hiện trong notebook

Notebook được tổ chức theo các bước sau:

1. Đọc các file Markdown của template văn bản
2. Trích xuất metadata từ YAML frontmatter
3. Phân tích cấu trúc template
4. Đọc dataset Parquet
5. Kiểm tra schema của dataset
6. Phân tích dữ liệu mẫu
7. Thống kê cơ bản

Các bước này giúp nhóm dự án hiểu rõ cấu trúc dữ liệu trước khi tiến hành các bước xử lý tiếp theo.

## Hướng dẫn chạy notebook

Để chạy notebook này cần cài đặt các thư viện Python sau:

- pandas
- pyarrow
- pathlib
- pyyaml

Sau đó:

1. Cập nhật đường dẫn tới dataset nếu cần.
2. Chạy các cell theo thứ tự từ trên xuống.
3. Kiểm tra kết quả hiển thị ở từng bước.

Notebook được thiết kế để các thành viên trong nhóm có thể dễ dàng kiểm tra dữ liệu trước khi sử dụng trong pipeline của dự án.

## Lưu ý

- Các file Markdown chỉ là **mẫu văn bản (templates)**, không phải văn bản thực tế.
- Dataset Parquet chứa **văn bản hành chính và pháp luật thực tế**.
- Các placeholder trong template sẽ được sử dụng cho:

  - Sinh văn bản tự động
  - Trích xuất thông tin
  - So khớp với văn bản thực

Nếu phát hiện lỗi về schema hoặc dữ liệu thiếu, cần báo lại để chỉnh sửa dataset trước khi đưa vào các bước xử lý tiếp theo.

In [18]:
import pandas as pd
import numpy as np

legal_path = "dataset/legal_dataset.parquet"
form_path = "Forms/md"

# Kiểm tra các forms văn bản có sẵn (10 forms)

# Kiểm tra dataset văn bản pháp luật

### 2. Bộ dữ liệu văn bản pháp luật / hành chính (Parquet)

Bộ dữ liệu thứ hai chứa **các văn bản hành chính và pháp luật thực tế**.

Dữ liệu được lưu ở định dạng **Parquet** nhằm:

- Tối ưu dung lượng lưu trữ
- Tăng tốc độ đọc dữ liệu
- Phù hợp cho xử lý dữ liệu lớn

Dataset bao gồm các trường như:

- id
- ministry
- type
- date
- name
- chapter_id
- chapter_name
- article
- contebt

Trong notebook này chúng ta sẽ:

- Kiểm tra schema của dataset
- Xem một số bản ghi mẫu
- Kiểm tra giá trị thiếu (missing values)
- Phân tích sơ bộ dữ liệu

In [21]:
legal_df = pd.read_parquet(legal_path, engine = "fastparquet")
print(legal_df.columns)

Index(['id', 'ministry', 'type', 'name', 'chapter_id', 'chapter_name',
       'article', 'content'],
      dtype='object')


In [22]:
unique_legal_df = legal_df['type'].value_counts().reset_index()
unique_legal_df.columns = ['Loại văn bản', 'Số lượng']
print(unique_legal_df.to_markdown(index=False))

| Loại văn bản                                                                   |   Số lượng |
|:-------------------------------------------------------------------------------|-----------:|
| THÔNG TƯ                                                                       |     147670 |
| QUYẾT ĐỊNH                                                                     |     130574 |
| NGHỊ QUYẾT                                                                     |      90768 |
| NGHỊ ĐỊNH                                                                      |      82882 |
| LUẬT                                                                           |      32609 |
| QUYẾT                                                                          |       8041 |
| ĐỊNH                                                                           |            |
| none                                                                           |       6660 |
| NGHỊ                                  

In [23]:
#Print first row of legal_df
print(legal_df.iloc[0])

id                                     01/2014/NQLT/CP-UBTƯMTTQVN
ministry        CHÍNH PHỦ - ỦY BAN TRUNG ƯƠNG MẶT TRẬN TỔ QUỐC...
type                                         NGHỊ QUYẾT LIÊN TỊCH
name            HƯỚNG DẪN PHỐI HỢP THỰC HIỆN MỘT SỐ QUY ĐỊNH C...
chapter_id                                               Chương I
chapter_name                                 NHỮNG QUY ĐỊNH CHUNG
article                                Điều 1. Phạm vi điều chỉnh
content         Nghị quyết liên tịch này hướng dẫn phối hợp th...
Name: 0, dtype: object


In [25]:
# Danh sách các văn bản "sống còn" cho soạn thảo hành chính và RAG
# Key: Tên hiển thị, Value: Regex để tìm kiếm
important_laws = {
    "Nghị định 30/2020/NĐ-CP (Văn thư)": r"30/2020/NĐ-CP",
    "Bộ luật Lao động 2019": r"45/2019/QH14|Lao động 2019",
    "Luật Cán bộ, công chức": r"Luật Cán bộ, công chức",
    "Luật Viên chức": r"Luật Viên chức",
    "Luật Ban hành văn bản QPPL": r"80/2015/QH13|15/2020/QH14",
    "Luật Lưu trữ": r"01/2011/QH13",
    "Luật Tổ chức Chính phủ": r"Luật Tổ chức Chính phủ",
    "Luật Tổ chức chính quyền địa phương": r"Luật Tổ chức chính quyền địa phương"
}

def check_legal_coverage(legal_df):
    print(f"--- Đang bắt đầu phân tích: ---")
    
    # 1. Chỉ đọc các cột cần thiết để tối ưu bộ nhớ
    cols = ['id', 'name', 'type', 'ministry']
    try:
        # Đọc dữ liệu (Chỉ lấy metadata để kiểm tra cho nhanh)
        df = legal_df
        print(f"✅ Đã tải thành công {len(df):,} bản ghi.\n")
    except Exception as e:
        print(f"❌ Lỗi khi đọc file: {e}")
        return

    # 2. Kiểm tra danh mục văn bản quan trọng
    results = []
    print("Đang quét các văn bản cốt lõi...")
    
    for label, pattern in important_laws.items():
        # Kiểm tra đồng thời trong cột 'id' và 'name'
        found_mask = (
            df['id'].str.contains(pattern, case=False, na=False) | 
            df['name'].str.contains(pattern, case=False, na=False)
        )
        count = found_mask.sum()
        status = "✅ CÓ" if count > 0 else "❌ THIẾU"
        results.append({
            "Văn bản pháp lý": label,
            "Trạng thái": status,
            "Số lượng dòng (Articles)": count
        })

    # In báo cáo dưới dạng bảng
    report_df = pd.DataFrame(results)
    print("\n" + "="*60)
    print("BÁO CÁO ĐỘ PHỦ DỮ LIỆU PHÁP LUẬT")
    print("="*60)
    print(report_df.to_string(index=False))
    print("="*60)

    # 3. Thống kê chuyên sâu về cơ cấu dữ liệu
    print("\nTOP 5 CƠ QUAN BAN HÀNH NHIỀU NHẤT:")
    print(df['ministry'].value_counts().head(5))

    print("\nTOP 5 LOẠI VĂN BẢN PHỔ BIẾN:")
    print(df['type'].value_counts().head(5))

if __name__ == "__main__":
    check_legal_coverage(legal_df)

--- Đang bắt đầu phân tích: ---
✅ Đã tải thành công 515,188 bản ghi.

Đang quét các văn bản cốt lõi...

BÁO CÁO ĐỘ PHỦ DỮ LIỆU PHÁP LUẬT
                    Văn bản pháp lý Trạng thái  Số lượng dòng (Articles)
  Nghị định 30/2020/NĐ-CP (Văn thư)       ✅ CÓ                        63
              Bộ luật Lao động 2019       ✅ CÓ                       220
             Luật Cán bộ, công chức       ✅ CÓ                       164
                     Luật Viên chức       ✅ CÓ                        48
         Luật Ban hành văn bản QPPL       ✅ CÓ                       180
                       Luật Lưu trữ       ✅ CÓ                        42
             Luật Tổ chức Chính phủ       ✅ CÓ                         4
Luật Tổ chức chính quyền địa phương       ✅ CÓ                         4

TOP 5 CƠ QUAN BAN HÀNH NHIỀU NHẤT:
ministry
CHÍNH PHỦ                85021
QUỐC HỘI                 32538
BỘ TÀI CHÍNH             27455
BỘ GIAO THÔNG VẬN TẢI    14169
BỘ CÔNG THƯƠNG           11106
Name: 